# 01 · 속도 재계산 규칙 정하기 (탐색)

전처리 첫 단계. 제공 `Vehicle_Speed`는 정지·저속 구간이 결측이라 좌표에서 속도를 다시 계산한다.
이 노트북에서 **규칙을 정하고**, 그 규칙을 `src/04_speed.py`로 800개 세션에 적용한다.
샘플 1개 세션으로 탐색.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트(Windows)
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs', exist_ok=True)

f = '../data/interim/2022-10-04_A/2022-10-04_A_AM1.csv'
df = pd.read_csv(f, low_memory=False)
df['sp'] = pd.to_numeric(df['Vehicle_Speed'], errors='coerce')   # 제공 속도
df['t'] = pd.to_timedelta(df['Local_Time']).dt.total_seconds()
print('세션 행수', df.shape[0], '| 차량수', df['Vehicle_ID'].nunique())

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트(Windows)
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('../outputs/speed', exist_ok=True)

f = '../data/interim/2022-10-04_A/2022-10-04_A_AM1.csv'
df = pd.read_csv(f, low_memory=False)
df['sp'] = pd.to_numeric(df['Vehicle_Speed'], errors='coerce')   # 제공 속도
df['t'] = pd.to_timedelta(df['Local_Time']).dt.total_seconds()
print('세션 행수', df.shape[0], '| 차량수', df['Vehicle_ID'].nunique())

In [ ]:
# 이동 차량 하나 선택 (제공속도 중앙>25km/h, 프레임 충분)
g = df.groupby('Vehicle_ID').agg(n=('sp', 'size'), spmed=('sp', 'median')).query('n>300 and spmed>25')
vid = g.sort_values('n', ascending=False).index[0]

v = df[df['Vehicle_ID'] == vid].sort_values('t').reset_index(drop=True)
v['dt'] = v['t'].diff()
v['d_local'] = np.sqrt(v['Local_X'].diff()**2 + v['Local_Y'].diff()**2)
v['d_ortho'] = np.sqrt(v['Ortho_X'].diff()**2 + v['Ortho_Y'].diff()**2)

print(f'차량 {vid}: 프레임 {len(v)}개')
print(f'프레임간 이동거리 중앙 — Local {v.d_local.median():.3f} m / Ortho {v.d_ortho.median():.3f} m')
print(f'Local 속도 중앙 {(v.d_local / v.dt * 3.6).median():.1f} km/h vs 제공 {v.sp.median():.1f} km/h')
print('→ Local이 제공 속도와 맞음. Ortho는 스케일이 달라 사용하지 않는다.')

## 2) 재계산 속도 검증 — 제공 속도와 일치하나
`Local_X/Y`로 계산한 속도가 제공 속도와 맞는지 산점도로 확인. 세그먼트 경계(Δt>0.1초)는 위치 점프라 제외.

In [ ]:
v['recompute'] = np.where((v['dt'] > 0) & (v['dt'] <= 0.1), v['d_local'] / v['dt'] * 3.6, np.nan)   # km/h, 경계·중복 제외

comp = v.dropna(subset=['recompute', 'sp'])
r = comp['recompute'].corr(comp['sp'])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(comp['sp'], comp['recompute'], s=6, alpha=0.3, color='#4C78A8')
lim = [0, max(comp['sp'].max(), comp['recompute'].max())]
ax.plot(lim, lim, 'r--', lw=1, label='y=x')
ax.set_xlabel('제공 속도 (km/h)')
ax.set_ylabel('재계산 속도 (km/h)')
ax.set_title(f'재계산 vs 제공 — r={r:.3f} (일치)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/speed/speed_recompute_vs_provided.png', dpi=120)
plt.show()

## 3) 정지 구간 — 제공은 비고, 재계산은 채운다
제공 속도는 정지·저속에서 NaN이라 선이 끊긴다. 재계산은 그 구간을 채운다.

In [ ]:
t0 = v['t'].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(v['t'] - t0, v['recompute'], color='#4C78A8', lw=0.9, label='재계산(Local)')
ax.plot(v['t'] - t0, v['sp'], color='crimson', lw=1.3, label='제공(정지 구간 NaN)')
ax.set_xlabel('시간 (초)')
ax.set_ylabel('속도 (km/h)')
ax.set_title(f'차량 {vid} 속도 프로파일 — 재계산이 정지 구간(제공 NaN)을 채움')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/speed/speed_profile.png', dpi=120)
plt.show()

## 4) 노이즈와 평활
프레임 단위 재계산은 떨린다. 이동 중앙값 평활로 완화하되 실제 가감속은 보존한다.

In [ ]:
v['smooth'] = v['recompute'].rolling(11, center=True, min_periods=1).median()   # 11프레임(~0.37초)

w2 = v.iloc[100:400]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(w2['t'] - w2['t'].iloc[0], w2['recompute'], color='#bbbbbb', lw=0.8, label='원시')
ax.plot(w2['t'] - w2['t'].iloc[0], w2['smooth'], color='#4C78A8', lw=1.6, label='평활(11프레임 중앙)')
ax.set_xlabel('시간 (초)')
ax.set_ylabel('속도 (km/h)')
ax.set_title(f'평활 효과 — 원시 std {v.recompute.std():.1f} → 평활 {v.smooth.std():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/speed/speed_smoothing.png', dpi=120)
plt.show()

## 정해진 규칙 → `src/04_speed.py`

- **좌표**: `Local_X/Y` (미터, EPSG:5186). `Ortho`는 정사영상 픽셀이라 안 씀.
- **속도** = √(ΔX²+ΔY²)/Δt × 3.6 (km/h), 차량별·시간순.
- **Δt>0.1초(세그먼트 경계) 또는 Δt=0(중복 타임스탬프)**는 → 그 프레임 속도 NaN.
- **평활**: 이동 중앙값(창 크기는 민감도로 검토).
- 제공 `Vehicle_Speed`는 버리고 재계산값 사용. (검증: r≈0.99 일치 — 제공값도 같은 좌표에서 가우시안 평활로 나와 순환적)